# Experiment: change-impact test selection vs baselines

This notebook measures the Mithuna selector against two baselines on a **realistic multilingual portal dataset** (generated, seed 42 — not live citizen data).

| Goal | Target |
| --- | --- |
| Baseline suite | Run all 44 org tests, **1095 s** |
| Time reduction | **≥ 35%** average vs run-all |
| Safety | **Recall 1.0** on ground-truth affected tests (never skip a test that would catch the change) |

Reproduce:

```bash
python scripts/generate_data.py
python experiments/run_experiment.py
```

Then re-run the cells below. Numbers in the markdown summary match the last `experiments/results.json`.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "experiments").exists() else Path.cwd().parent
results_path = ROOT / "experiments" / "results.json"
if not results_path.exists():
    import sys
    sys.path.insert(0, str(ROOT))
    from experiments.run_experiment import main
    main()
report = json.loads(results_path.read_text(encoding="utf-8"))
report["averages"], report["targets"]

## Baselines

1. **Run-all** — every test in `ministry_interior`. This is today’s field default when teams do not trust selection.
2. **File-pattern** — keep tests whose name/tags share a path token with the change, plus smoke/security. Cheap, common in CI scripts, and **unsafe** when impact is transitive (auth → benefits application).
3. **Selector** — coverage + dependency expansion + locale rules + failure history, fail-closed when a source is missing.

In [ ]:
rows = []
for change in report["changes"]:
    rows.append({
        "change_id": change["change_id"],
        "kind": change["kind"],
        "all_s": change["run_all"]["suite_seconds"],
        "pattern_s": change["pattern"]["suite_seconds"],
        "selector_s": change["selector"]["suite_seconds"],
        "selector_save_pct": change["selector"]["time_saved_pct"],
        "pattern_recall": change["pattern"]["recall"],
        "selector_recall": change["selector"]["recall"],
        "pattern_missed": change["pattern"]["missed_affected"],
    })
for row in rows:
    print(
        f"{row['change_id']:28} kind={row['kind']:8} "
        f"all={row['all_s']:.0f}s pattern={row['pattern_s']:.0f}s "
        f"sel={row['selector_s']:.0f}s save={row['selector_save_pct']}% "
        f"rec_p={row['pattern_recall']} rec_s={row['selector_recall']}"
    )

## Measured results (last run)

| Method | Average suite time | Average time saved | Average recall |
| --- | --- | --- | --- |
| Run-all | 1095.0 s | 0% | 1.000 |
| File-pattern | 348.8 s | 68.2% | **0.793** |
| **Selector** | **624.0 s** | **43.0%** | **1.000** |

The 35% time-reduction goal is met. Pattern matching “wins” on time and **fails the safety goal**.

**Content vs code.** Hindi benefits copy (`chg_hi_benefits_copy`): selector **373 s** (66% saved), both affected tests kept. Auth session (`chg_auth_session`): selector **~15%** saved — the graph correctly keeps a wide radius; pattern saved 80% and **missed** `t_int_benefits_flow`.

In [ ]:
print("Selector misses (must be empty):", report["error_analysis"]["selector_miss_count"])
print("Pattern misses by change:")
for item in report["error_analysis"]["pattern_misses"]:
    print(f"  {item['change_id']}: {item['missed']}")

## Error analysis

**Selector:** zero skipped tests that the ground-truth map marked as affected (the test that would fail if the change were buggy).

**Pattern baseline failures (would have shipped a miss):**

| Change | Missed tests | Why pattern failed |
| --- | --- | --- |
| `chg_auth_session` | `t_int_benefits_flow` | Name has “benefits”, file is `session.py` |
| `chg_payments_gateway` | `t_int_benefits_flow` | Application depends on gateway; names do not match |
| `chg_i18n_fallback` | locale common tests, notify + a11y | Formatter/fallback fan-out is not in the test name |
| `chg_schema_profile` | `t_unit_upload` | Upload depends on profile via the graph |
| `chg_search_typo` | `t_int_benefits_flow` | Search is a benefits dependency |

These are exactly the cases a government portal hits: shared modules, not folder-shaped tests.

**False positives (extra tests run)** are expected. Smoke/security are policy keeps. Over-keeping is preferred to under-keeping.

In [ ]:
print("Auth change under source outages (selector suite seconds and recall)")
for row in report["degradation"]:
    sel = row["selector"]
    print(
        f"missing={row['missing'] or ['none']:20} "
        f"threshold={row['skip_threshold']} "
        f"seconds={sel['suite_seconds']} "
        f"recall={sel['recall']} "
        f"missed={sel['missed_affected']}"
    )

## Graceful degradation

On `chg_auth_session`:

| Missing source | Skip threshold | Selector seconds | Recall |
| --- | --- | --- | --- |
| None | 0.18 | ~926 s | 1.0 |
| Coverage | 0.30 | more tests kept | 1.0 |
| Dependencies | 0.28 | integration/e2e retained by policy | 1.0 |
| Failure history | slightly higher | 1.0 |

Time savings shrink when data is incomplete. That is the operational contract: **the pipeline stays green and conservative**, it does not invent skips.

## How to read a skip

Open any `skipped_examples` entry in `results.json`, or call `GET /v1/explain/{change_id}/{test_id}`.

A skip is trustworthy when:

1. `impact_score` < `skip_threshold`
2. `sources_used` lists what was actually loaded
3. Reasons show **no** coverage overlap, **no** dependency path, and either no locale overlap or an explicit other-locale content rule
4. The test is not in `ground_truth.json` for that change

If you cannot explain the skip in one sentence, do not apply it in CI (role `auditor` already cannot).